# Instagram Product Data Normalization & Keyword Cleaning

이 노트북은 인스타그램 제품 데이터를 정제하여 분석 가능한 형태로 가공하는 파이프라인입니다.
원본 데이터의 기호를 보존한 상태에서 필터링을 먼저 수행한 후 정규화를 진행합니다.

## Step 1. 환경 설정 및 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import os
import re
import json
import ast
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

In [2]:
BASE_DIR = r'C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework'
FILE_PATH = os.path.join(BASE_DIR, 'data', 'processed', '편의점_instagram', 'merged_instagram_products_final.xlsx')

df = pd.read_excel(FILE_PATH)
print(f"Original Data Shape: {df.shape}")
df.head()

Original Data Shape: (3776, 10)


,p_name,p_price,p_cap,p_attrs,brand,type,date,body,url,source_file
0,생오렌지 하이볼,4500.0,500ml,"['생오렌지', '오렌지', '과즙', '하이볼', '묶음할인', '간편음료']",CU,단일,2025-01-05 00:00:00,생과일 하이볼 시리즈 7탄\n국내 최초! 생오렌지 하이볼\n\n생오렌지 슬라이스가 ...,https://www.instagram.com/p/DEcCgJOu2bz/,CU_단일
1,미나리 우곡생주,11000.0,750ml,"['미나리', '참외', '쌀', '전통주', '막걸리', '배혜정']",CU,단일,2025-01-06 00:00:00,우곡생주에 미나리와 참외 플레이버로\n향긋함과 은은한 단맛을 더하고\n100% 쌀과...,https://www.instagram.com/p/DEenQi5PT71/,CU_단일
2,삼각김밥 전 품목,0.0,NaN,"['삼각김밥', '신한 SOL페이', '할인', '기간한정', '월', '연초']",CU,단일,2025-01-06 00:00:00,1월에도 3.6.5를 기억한다면?!\n3각김밥 🍙 6시간 동안 5백원 할인😘\n\n...,https://www.instagram.com/p/DEeibRVS9aK/,CU_단일
3,get커피 아이스아메리카노 XL,0.0,XL,"['커피', '아메리카노', '음료', '카카오페이', '타임할인', '결제할인',...",CU,단일,2025-01-06 00:00:00,1월에도 아침엔 get커피와 함께🌞\n\n☕get커피 아이스아메리카노 XL☕\n\n...,https://www.instagram.com/p/DEeTIMYSZ8W/,CU_단일
4,신년맞이떡만둣국,5700.0,NaN,"['떡만둣국', '국물/국', '간편식', '김가루', '떡', '만두', '단독'...",CU,단일,2025-01-06 00:00:00,2025년도 CU와 함께\n\n신년맞이 떡만둣국 출시\n비법 레시피로 만든 진한 육...,https://www.instagram.com/p/DEd3KhMpshs/?img_i...,CU_단일


## Step 2. 기초 속성 파싱 (Attribute Extraction)

In [3]:
def safe_parse_attrs(val):
    if pd.isna(val) or val == '[]' or val == '': return []
    try:
        if isinstance(val, str):
            try: return json.loads(val)
            except: return ast.literal_eval(val)
        return val
    except: return []

df['p_attrs_list'] = df['p_attrs'].apply(safe_parse_attrs)

STOPWORDS_CATEGORIZED = {
    'Contextual': {'이벤트', '참여', '댓글', '팔로우', '당첨', '경품', '사전예약', '선착순', '증정', '할인', '행사', '원플러스원', '투플러스원', '1+1', '2+1', '공식', '계정', '어플', '앱', '포켓CU', '우리동네GS', '없음'},
    'Marketing': {'신상', '신제품', 'NEW', '추천', '인기', '대박', '출시', '한정판', '한정', '단독', '주목', '달려가세요', '쟁여두세요', '필수', '박스', '기획', '패키지', '에디션', '컬렉션', '시리즈'},
    'Channel': {'세븐일레븐', 'CU', 'GS25', '씨유', '지에스', '편의점', '세븐', '편의점신상'},
    'Subjective': {'맛있다', '예쁘다', '좋아요', '강추', '비주얼', '꿀맛', '존맛', '미쳤다', '역대급', 'JMT'}
}
ALL_STOPWORDS = set().union(*STOPWORDS_CATEGORIZED.values())

def clean_keywords(attrs):
    if not isinstance(attrs, list): return []
    cleaned = []
    for kw in attrs:
        kw = str(kw).replace(" ", "").strip()
        if kw in ALL_STOPWORDS or len(kw) <= 1: continue
        if re.search(r'\d+(ml|g|kg|l|개|입|봉|팩|병|캔)', kw, re.I): continue
        cleaned.append(kw)
    return list(set(cleaned))

df['p_attrs_cleaned'] = df['p_attrs_list'].apply(clean_keywords)

## Step 3. 원본 기준 필터링 (Raw Filtering)

In [4]:
contains_exclude = ['오늘의 메뉴', 'PBICK', '거강기능식품', '장건강', '&']
exact_exclude = ['탄산음료', '삼각김밥', '도시락', '김밥', '주먹밥', '샌드위치', '햄버거', '와인', '스프린트', '청년다방', '김치']
mask_contains = df['p_name'].str.contains('|'.join(contains_exclude), na=False, case=False)
mask_exact = df['p_name'].isin(exact_exclude)
combined_mask = mask_contains | mask_exact
df = df[~combined_mask].reset_index(drop=True)

## Step 4. 제품명 정규화 (Normalization)

In [5]:
def normalize_product_name(name):
    if not isinstance(name, str): return name
    name = re.sub(r'\[.*?\]|\(.*?\)', '', name)
    noise_words = ['신상', '한정판', '2\\+1', '1\\+1', '증정', '단독', '출시', 'NEW']
    for word in noise_words: name = name.replace(word, '')
    name = re.sub(r'[^a-zA-Z0-9가-힣\\s+]', ' ', name)
    return re.sub(r'\\s+', ' ', name).strip()

df['p_name_clean'] = df['p_name'].apply(normalize_product_name)

In [6]:
def merge_unique_keywords(series):
    merged = []
    for k_list in series:
        if isinstance(k_list, list): merged.extend(k_list)
    return list(set(merged))

agg_dict = {col: 'first' for col in df.columns if col not in ['p_name', 'p_attrs_cleaned']}
agg_dict['p_attrs_cleaned'] = merge_unique_keywords
df = df.groupby('p_name', as_index=False).agg(agg_dict)

## Step 5. 심층 키워드 정제 및 노이즈 탐색

In [7]:
REMOVE_PARTS = ['각종', '의맛']
STRICT_TARGETS = ['경주', '고소', '짭잘', '골든', '공부', '안유성']

def surgical_clean(attr_list):
    if not isinstance(attr_list, list): return []
    new_attrs = []
    for kw in attr_list:
        kw = str(kw).strip()
        matched_strict = False
        for target in STRICT_TARGETS:
            if target in kw: 
                new_attrs.append(target)
                matched_strict = True
                break
        if matched_strict: continue
        cleaned_kw = kw
        for part in REMOVE_PARTS: cleaned_kw = cleaned_kw.replace(part, '')
        if len(cleaned_kw.strip()) > 1: new_attrs.append(cleaned_kw.strip())
    return list(set(new_attrs))

df['p_attrs_cleaned'] = df['p_attrs_cleaned'].apply(surgical_clean)

## Step 5.4. LLM 복합어 분석용 중간 저장 (pre_smart_clean.parquet)

In [8]:
# smart_clean_runner.py 실행 전 필수 입력 파일 생성
PRE_CLEAN_PATH = os.path.join(BASE_DIR, 'data', 'processed', '편의점_instagram', 'pre_smart_clean.parquet')

df_pre = df[['p_name', 'p_attrs_cleaned']].copy()
df_pre['p_attrs_rescued'] = [[] for _ in range(len(df_pre))]
df_pre.to_parquet(PRE_CLEAN_PATH, index=False, engine='pyarrow')

print(f'저장 완료: {PRE_CLEAN_PATH}')
print(f'  제품 수: {len(df_pre)}개')
print(f'  평균 키워드 수: {df_pre["p_attrs_cleaned"].apply(len).mean():.1f}개/제품')
print(f'>> 이제 터미널에서 실행:')
print(f'   python src/data_builder/smart_clean_runner.py')

저장 완료: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\편의점_instagram\pre_smart_clean.parquet
  제품 수: 2930개
  평균 키워드 수: 6.5개/제품
>> 이제 터미널에서 실행:
   python src/data_builder/smart_clean_runner.py


## Step 5.5. LLM 복합어 정제 결과 분석 (smart_clean_result.xlsx)

In [9]:
RESULT_XLSX    = os.path.join(BASE_DIR, 'data', 'processed', '편의점_instagram', 'smart_clean_result.xlsx')
COMPARE_XLSX   = os.path.join(BASE_DIR, 'eda', 'df_compare_keywords.xlsx')
PRE_CLEAN_PATH = os.path.join(BASE_DIR, 'data', 'processed', '편의점_instagram', 'pre_smart_clean.parquet')

if not os.path.exists(RESULT_XLSX):
    print("smart_clean_result.xlsx 파일이 없습니다. smart_clean_runner.py를 먼저 실행하세요.")
elif not os.path.exists(PRE_CLEAN_PATH):
    print("pre_smart_clean.parquet 파일이 없습니다. Step 5.4부터 다시 실행하세요.")
else:
    # p_attrs_rescued 컬럼 충돌 방지 — p_name, p_attrs_cleaned 만 사용
    df_base = pd.read_parquet(PRE_CLEAN_PATH)[['p_name', 'p_attrs_cleaned']]

    df_res = pd.read_excel(RESULT_XLSX)

    def to_list(val):
        if pd.isna(val) or str(val).strip() == '': return []
        return [w.strip() for w in str(val).split(',')]

    df_res['p_attrs_confirmed'] = df_res['확정_키워드'].apply(to_list)
    df_res['p_attrs_rescued']   = df_res['생존후보_키워드'].apply(to_list)

    df_merged = df_base.merge(
        df_res[['제품명', 'p_attrs_confirmed', 'p_attrs_rescued']],
        left_on='p_name', right_on='제품명', how='left'
    )
    df_merged['p_attrs_confirmed'] = df_merged['p_attrs_confirmed'].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df_merged['p_attrs_rescued'] = df_merged['p_attrs_rescued'].apply(
        lambda x: x if isinstance(x, list) else []
    )

    df_compare = pd.DataFrame({
        '제품명':          df_merged['p_name'],
        '이전 키워드':     df_merged['p_attrs_cleaned'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
        '확정 키워드':     df_merged['p_attrs_confirmed'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
        '생존후보_키워드': df_merged['p_attrs_rescued'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
    })

    display(df_compare.head(20))
    print(f'\n총 {len(df_compare)}개 제품 | LLM 정제 적용: {(df_compare["확정 키워드"] != "").sum()}개')

    # df가 메모리에 있으면 이후 Step 6 연속 실행을 위해 컬럼 추가
    try:
        _idx = df_merged.set_index('p_name')
        df['p_attrs_before']    = df['p_attrs_cleaned']
        df['p_attrs_confirmed'] = df['p_name'].map(_idx['p_attrs_confirmed']).apply(
            lambda x: x if isinstance(x, list) else []
        )
        df['p_attrs_rescued']   = df['p_name'].map(_idx['p_attrs_rescued']).apply(
            lambda x: x if isinstance(x, list) else []
        )
        df['p_attrs_cleaned']   = df['p_attrs_confirmed']
    except NameError:
        pass

    # 검수용 파일 자동 저장
    df_compare.to_excel(COMPARE_XLSX, index=False)
    print(f'\n검수용 파일 저장: {COMPARE_XLSX}')
    print('  검수 완료 후 00_product_keyword_pipeline.ipynb를 실행하세요.')

,제품명,이전 키워드,확정 키워드,생존후보_키워드
0,100%두리안바,,"간식, 두리안, 스낵, 프리미엄, 한정판매",
1,1000트위스트,,"골든, 디저트, 떡볶이, 로제, 마요, 매콤, 밀크, 반찬, 비빔, 서울, 소다, ...","전주, 팝"
2,1664블랑캔,,"맥주, 묶음할인, 쟁여두기",
3,1865청뱀띠에디션,,"1865, 과일, 바닐라, 시즌, 와인, 청뱀띠, 특별할인",한정
4,1988 버거,,"1988, 도시락, 시즌, 야식, 코카, 콜라, 콤보","한정, 할인"
5,21 ROCS 칵테일,,"RTD, 모스코뮬, 모히또, 베리, 와인, 칵테일, 탄산","베이스, 스트로"
6,290 캡슐커피,,"오피스, 카페, 커피","캡슐, 홈"
7,2분컵(고깃집맛된장국),,"가성비, 간편, 국물, 된장, 식사","국, 대용"
8,2분컵(한우미역국),,"가성비, 간편, 국물, 미역, 식사","국, 대용"
9,2분컵(한우소고기무국),,"고기, 국물, 무국, 신상품, 야식, 칼칼, 한끼","가벼운, 한우소"



총 2930개 제품 | LLM 정제 적용: 2928개

검수용 파일 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\eda\df_compare_keywords.xlsx
  검수 완료 후 00_product_keyword_pipeline.ipynb를 실행하세요.


# Step 6. 키워드 대조 질적분석 및 정밀 복구

In [ ]:
COMPARE_XLSX = os.path.join(BASE_DIR, 'eda', 'df_compare_keywords.xlsx')
if os.path.exists(COMPARE_XLSX):
    df_qual = pd.read_excel(COMPARE_XLSX)
    print(f"📊 질적 분석 대조 데이터 로드 완료: {len(df_qual)}개 제품")

In [ ]:
# 1. 복구 키워드 리스트 선언
MUST_RESTORE_FROM_BEFORE = [
    '가나디', '불고기', '물만두', '매드포갈릭', '마른안주', '파인애플', '게맛살', '샌드위치', '콩나물', '닭고기', '군만두', '시리얼', '알룰로스', '급식대가', '베이커리', '프랑스', '이모카세', '떡갈비', '피넛버터', '라즈베리', '오리온', '오징어게임', '올리브유', '보양식', '아이스크림', '당충전', '흑백요리사', '칼국수', '디저트39', '편다이닝', '이웃집통통이', '고독한미식가더무비', '아메리카노', '우유니소금', '짱구는못말려', '콘치즈', '포켓몬스터', '버터베어', '시트러스', '애플하우스', '역전우동', '황치즈', '리얼프라이스', '에드워드리', '밀크티', '차돌박이', '타코야끼', '투다리', '소주전쟁', '앙버터', '아사이볼'
]
MUST_RESCUE_WORDS = ['밥', '감귤', '쏘이', '궁채', '퓨전', '야식', '제철', '술', '유튜버', '봄', '빵']
INSPECT_EXACT = ['세일', '할인', '수건', '빵', '두산', '베어스', '티', '아이스', '1인', '릴로&스티치', '삼각', '셰프', '데이']
INSPECT_REGEX = [r'.*베리$', r'.*고기$', r'.*가루$']

def restore_logic(row):
    base = list(row['p_attrs_confirmed']) if isinstance(row['p_attrs_confirmed'], list) else []
    before = list(row['p_attrs_before']) if isinstance(row['p_attrs_before'], list) else []
    rescued = list(row['p_attrs_rescued']) if isinstance(row['p_attrs_rescued'], list) else []
    for kw in MUST_RESTORE_FROM_BEFORE: 
        if kw in before: base.append(kw)
    for kw in MUST_RESCUE_WORDS: 
        if kw in before or kw in rescued: base.append(kw)
    return list(set(base))

df['p_attrs_cleaned'] = df.apply(restore_logic, axis=1)

In [ ]:
# 패턴 검토 EDA 테이블 생성
inspection_rows = []
for idx, row in df.iterrows():
    all_targets = list(set((row['p_attrs_before'] if isinstance(row['p_attrs_before'], list) else []) + (row['p_attrs_rescued'] if isinstance(row['p_attrs_rescued'], list) else [])))
    matched = []
    for kw in all_targets:
        for target in INSPECT_EXACT:
            if target in kw: matched.append(f"{kw}({target})")
        for pattern in INSPECT_REGEX:
            if re.search(pattern, kw): matched.append(f"{kw}(Regex)")
    if matched:
        inspection_rows.append({'제품명': row['p_name'], '매칭패턴': ", ".join(list(set(matched))), '이전': ", ".join(row['p_attrs_before']), '확정': ", ".join(row['p_attrs_confirmed']), '후보': ", ".join(row['p_attrs_rescued'])})

unique_matched_kws = set()
for row in inspection_rows:
    for p in row['매칭패턴'].split(', '):
        unique_matched_kws.add(p.split('(')[0])

print(f"📊 패턴 검토 대상 제품: {len(inspection_rows)}개")
print("\n🔍 [패턴 매칭 유니크 키워드 목록]")
print(", ".join(sorted(list(unique_matched_kws))))

df_inspection = pd.DataFrame(inspection_rows)

In [ ]:
# 1. 검토할 패턴 정의
INSPECT_EXACT = [
    '세일', '할인', '수건', '빵', '두산', '베어스', '망곰','티', '아이스', '배달',
    '1인', '릴로&스티치', '삼각', '셰프', '데이'
]
INSPECT_REGEX = [r'.*베리$', r'.*고기$', r'.*가루$']

def inspect_keywords(df):
    all_kws = set()
    for _, row in df.iterrows():
        before = row['p_attrs_before'] if isinstance(row['p_attrs_before'], list) else []
        rescued = row['p_attrs_rescued'] if isinstance(row['p_attrs_rescued'], list) else []
        all_kws.update(before)
        all_kws.update(rescued)

    pattern_map = {p: set() for p in INSPECT_EXACT + INSPECT_REGEX}

    for kw in all_kws:
        kw_str = str(kw)
        for p in INSPECT_EXACT:
            if p in kw_str:
                pattern_map[p].add(kw_str)
        for p in INSPECT_REGEX:
            if re.search(p, kw_str):
                pattern_map[p].add(kw_str)

    print("🔍 [패턴별 매칭 키워드 리스트]\n")
    for pattern, matched_list in pattern_map.items():
        if matched_list:
            sorted_list = sorted(list(matched_list))
            print(f"📌 패턴: [{pattern}]")
            print(f"   ㄴ 키워드({len(sorted_list)}개): {', '.join(sorted_list)}")
            print("-" * 50)
        else:
            print(f"📌 패턴: [{pattern}] -> 매칭된 키워드 없음")

## Step 5.7. 프로모션 키워드 정규화 (Promotion Keyword Normalization)

In [ ]:
# 프로모션 맵 확정
categorized_dict = {
    '0104 : 콤보할인': [
        '콤보할인', '2종콤보할인', '콤보구매할인', '세트할인',
        '콤보행사', '할인콤보', 
        '콤보'
    ],
    '1+1': ['1+1', '1+1행사'],
    '2+2': ['2+2'],
    '3+1': ['3+1'],
    '0107 : 묶음할인(구간)': [
        '0107 : 묶음할인(구간)',
        '묶음할인', '동반구매할인', '2+3', '3+3', '묶음행사', '교차행사',
        '교차가능', '교차구매', '묶음구매', '묶음판매',
        '증정', '증정품', '경품증정', '덤증정', '무료증정', '상시증정',
        '소스증정', '추가증정', '현장증정', '랜덤증정', '랜덤씰증정',
        '도시락구매시증정', '2종구매시증정', '2종구매증정'
    ],
    '0301 : 구독행사': [
        '0301 : 구독행사',
        '구독할인'
    ],
    '0205 : 장바구니할인': [
        '0205 : 장바구니할인',
        '결제할인', '결제조건할인', '결제혜택', '첫결제혜택', 'VVIP혜택',
        '카드할인', '비씨카드할인',
        '카카오페이', '카카오페이머니할인', '카카오페이머니10%할인', '카카오페이20%페이백',
        'GSPay할인', 'GSPay결제혜택',
        '네이버페이결제', '당근페이', '당근페이프로모션', '토스페이결제', '잇페이짱',
        '멤버십할인', 'T멤버십할인', 
        '멤버십신규가입', '신규가입이벤트', '유플투쁠',
        '할인쿠폰', '30%할인쿠폰증정', '소비쿠폰', '쿠폰',
        '중복할인', 'CU페이행사', 'QR할인', '혜택가'
    ],
    '0106 : 단품할인': [
        '0106 : 단품할인',
        '단품할인행사',
        '90%할인', '50%할인', '45%할인', '40%할인', '30%할인', '25%할인', '20%할인', '10%할인',
        '최대2,000원할인', '2,900원할인', '2100원할인', '1500원할인', '600원할인', '990원행사',
        '특가할인', '초특가할인', '할인특가', '할인가', '가격할인', '할인',
        '할인행사', '상시할인', '타임할인', '시간할인', '모닝할인', '연말할인',
        '오프라인할인', '추가할인',
        '정기행사', '상시행사', '증정행사', 
        '댓글이벤트', '이벤트진행중', '이벤트참여', '신규런칭',
        '세일', '초특가세일', '초특갓세일', '코리안세일페스타', '타임세일',
        '페이백', '20%페이백', '50%페이백'
    ],
}

kw_to_promo_code = {}
for code, kw_list in categorized_dict.items():
    for kw in kw_list:
        kw_to_promo_code[kw] = code

all_mapped_kws = set(kw_to_promo_code.keys())

PROMO_PATTERNS = [
    r'할인', r'세일', r'행사', r'쿠폰', r'혜택', r'증정',
    r'구독', r'페이', r'\d\+\d', r'1\+1', r'2\+1', r'플러스원'
]

all_dataset_kws = set()
for col in ['p_attrs_before', 'p_attrs_rescued', 'p_attrs_confirmed']:
    for kw_list in df[col]:
        if isinstance(kw_list, list):
            all_dataset_kws.update(str(k) for k in kw_list)

promo_kws_in_data = set()
for kw in all_dataset_kws:
    for pat in PROMO_PATTERNS:
        if re.search(pat, kw):
            promo_kws_in_data.add(kw)
            break

unmapped = promo_kws_in_data - all_mapped_kws

print(f"데이터 내 프로모션 키워드: {len(promo_kws_in_data)}개")
print(f"현재 맵 커버:             {len(promo_kws_in_data & all_mapped_kws)}개")
print(f"미매핑 키워드:            {len(unmapped)}개\n")
print("[미매핑 프로모션 키워드 목록]")
for kw in sorted(unmapped):
    print(f"  - {kw}")

## Step 5.7.5. IP 콜라보 키워드 탐색 EDA (Pre-processing Analysis)

In [ ]:
gs25_collab_expanded = {
    "게임_IP": {
        "블루 아카이브": ["블루아카", "블루아카이브", "몰루", "아로나"],
        "메이플스토리": ["메이플스토리", "메이플", "핑크빈", "예티", "주황버섯", "슬라임"],
        "승리의 여신: 니케": ["니케", "NIKKE", "시프트업"],
        "명조": ["명조", "워더링웨이브", "워더링"],
        "명일방주": ["명일방주", "로도스"],
    },
    "K팝_아이돌": {
        "세븐틴": ["세븐틴", "SVT", "SEVENTEEN", "캐럿"],
        "PLAVE": ["플레이브", "PLAVE", "예준", "노아", "밤비", "은호", "하민"],
    },
    "콘텐츠_IP": {
        "진격의 거인": ["진격의거인", "진격거", "리바이"],
        "흑백요리사": ["흑백요리사", "요리계급전쟁", "백수저", "흑수저", "급식대가", "나폴리맛피아", "이모카세", "안성재", "백종원"],
        "무신사": ["무신사", "무탠다드"],
        "몬치치": ["몬치치"],
        "EBSi": ["EBSi", "EBS"],
        "젼언니": ["젼언니", "멜로우빈"],
        "경동시장": ["경동시장"],
    }
}

cu_collab_expanded = {
    "게임_IP": {
        "닌텐도 피크민": ["피크민", "PIKMIN", "닌텐도"],
        "디아블로 IV": ["디아블로", "디아블로4", "성역"],
        "T1": ["T1", "페이커", "FAKER", "제오페구케"],
    },
    "K팝_아이돌": {
        "TXT": ["투모로우바이투게더", "TXT", "투바투", "MOA"],
        "지드래곤": ["피스마이너스원", "지드래곤", "GD", "권지용", "PMO", "데이지"],
        "스트레이키즈": ["스트레이키즈", "StrayKids", "스키즈", "SKZ"],
        "이세계아이돌": ["이세계아이돌", "이세돌", "아이네", "징버거", "릴파", "주르르", "고세구", "비챤"],
    },
    "캐릭터_IP": {
        "포켓몬스터": ["포켓몬", "포켓몬스터", "피카츄", "메타몽", "띠부씰"],
        "스누피": ["스누피", "피너츠", "찰리브라운"],
        "짱구": ["짱구", "짱구는못말려", "흰둥이", "짱아", "초코비", "못말려"],
        "가나디": ["가나디", "GANADI"],
        "티니핑": ["티니핑", "캐치티니핑", "하츄핑"],
        "빵빵이": ["빵빵이", "옥지"],
        "해리스트위드": ["해리스트위드"],
    },
    "F&B_브랜드": {
        "백종원": ["백종원", "백주부", "더본코리아"],
        "명륜진사갈비": ["명륜진사", "진사갈비"],
        "노티드": ["노티드", "Knotted", "슈가베어"],
        "비비고": ["비비고", "BIBIGO"],
        "오징어게임": ["오징어게임", "성기훈"],
    }
}

seven_collab_expanded = {
    "스포츠_IP": {
        "K리그": ["KLEAGUE", "K리그"],
        "KBL": ["KBL", "프로농구"],
        "이정후": ["이정후"],
        "해외축구": ["토트넘", "맨시티", "FIFA", "파니니"],
        "롯데자이언츠": ["롯데자이언츠"],
    },
    "셰프_맛집_IP": {
        "흑백요리사 셰프 - 최강록": ["최강록"],
        "흑백요리사 셰프 - 박은영": ["박은영"],
        "흑백요리사 셰프 - 안유성": ["안유성"],
        "흑백요리사 셰프 - 에드워드리": ["에드워드리"],
        "흑백요리사 셰프 - 이균": ["이균"],
        "F&B 브랜드": ["아티제", "홍콩제니", "부창제과", "온정돈까스", "디진다돈까스", "롯데리아", "마루짱", "뵈르뵈르", "청수당", "미노리키친", "푸하하소금빵", "디저트39", "장충동왕족발"],
    },
    "방송_콘텐츠_IP": {
        "셀럽/인플루언서": ["추성훈", "이장우", "하정우", "장민호", "이봉원", "히밥", "미미미누", "정희원"],
        "미디어 콘텐츠": ["티처스", "김부장이야기", "좀비딸", "유미의세포들"],
    },
    "캐릭터_애니_IP": {
        "산리오 / 키티": ["산리오", "헬로키티", "페코짱"],
        "디즈니 / 캐릭터": ["디즈니", "주토피아", "미키", "랏소", "테디베어", "리락쿠마", "오구", "헬로맨", "두햄빠"],
        "국내 인기 캐릭터": ["밸리곰", "라인프렌즈", "춘식이", "김잼작가", "키키블룸", "키키쿼카", "블루밍테일"],
        "애니메이션": ["귀멸의칼날"],
        "캐치티니핑": ["티니핑", "하츄핑"],
    },
    "K팝_아이돌": {
        "아이돌 그룹": ["케이팝데몬헌터스", "엔하이픈", "케플러", "트리플에스", "NCTWISH", "엔시티위시"],
    },
    "라이프스타일_IP": {
        "브랜드 콜라보": ["SK하이닉스", "이스타항공", "핫휠", "위글위글", "이나피스퀘어", "앙리마티스"],
    }
}

ip_search_map = {}
for collab_dict in [gs25_collab_expanded, cu_collab_expanded, seven_collab_expanded]:
    for cat, items in collab_dict.items():
        for ip_name, kws in items.items():
            for kw in kws:
                ip_search_map[kw] = {"ip": ip_name, "cat": cat}

CHEF_MULTI_MAP = {
    "최강록":    ["흑백요리사", "최강록"],
    "박은영":    ["흑백요리사", "박은영"],
    "안유성":    ["흑백요리사", "안유성"],
    "에드워드리": ["흑백요리사", "에드워드리"],
    "이균":      ["흑백요리사", "이균"],
}
for chef, multi_kws in CHEF_MULTI_MAP.items():
    ip_search_map[chef] = {"ip": multi_kws, "cat": "셰프_맛집_IP"}

## Step 5.8. 키워드 정규화 규칙 적용 및 확정키워드_최종 생성 (df_qual 기반)

In [ ]:
CONTAINS_REPLACE = [
    (r'수건',                                    ['수건', '케이크']),
    (r'망곰',                                    ['망곰']),
    (r'두산|베어스',                              ['KBO']),
    (r'요거트아이스정석|요거트아이스크림의정석',   ['요아정']),
]

EXACT_REPLACE = {
    '삼각':    ['삼각김밥'],
    '삼각김밥': ['삼각김밥'],
}

PATTERN_ALLOWLIST = {
    r'티':      {'노티드', '링티', '밀크티', '스파게티', '짜파게티', '아이스티', '캐치!티니핑', '티라미수', '블랙티'},
    r'아이스':  {'아이스크림', '아이스브륄레'},
    r'데이':    {'데이지에일', '데이트', '발렌타인데이', '블랙데이', '빼빼로데이', '화이트데이'},
    r'.*베리$': {'라즈베리', '블루라즈베리', '블루베리', '스트로베리', '아사이베리', '크랜베리'},
    r'.*고기$': {'불고기', '돼지고기', '닭고기', '머릿고기', '소고기', '쇠고기', '오리고기'},
    r'.*가루$': {'고춧가루', '김가루', '들깨가루', '콩가루'},
    r'배달':    {'배달', '픽업'},
    r'1인':     {'1인'},
    r'빵':      {'깨찰빵', '꽃빵', '맘모스빵', '미각제빵소', '붕어빵', '빵또아', '빵빵이', '소금빵', '소보로빵', '중화빵', '호빵'},
}

REMOVE_PATTERNS = [r'셰프']

In [ ]:
def parse_kw_col(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return [w.strip() for w in str(val).split(',') if w.strip()]

def apply_rules(kw, is_confirmed=False):
    kw = str(kw).strip()
    kw_clean = kw.replace(" ", "")

    if kw_clean in ip_search_map:
        ip_val = ip_search_map[kw_clean]['ip']
        if isinstance(ip_val, list):
            return ip_val
        return [ip_val]

    for pat in REMOVE_PATTERNS:
        if re.search(pat, kw):
            return []

    for pat, replacement in CONTAINS_REPLACE:
        if re.search(pat, kw):
            return replacement

    if kw in EXACT_REPLACE:
        return EXACT_REPLACE[kw]

    if kw in kw_to_promo_code:
        return [kw_to_promo_code[kw]]

    matched_any_pattern = False
    for pat, allowlist in PATTERN_ALLOWLIST.items():
        if re.search(pat, kw):
            matched_any_pattern = True
            if kw in allowlist:
                return [kw]

    if matched_any_pattern:
        return []

    if is_confirmed:
        return [kw]
    else:
        return []


def build_final_keywords(row):
    confirmed = parse_kw_col(row['확정 키워드'])
    before    = parse_kw_col(row['이전 키워드'])
    rescued   = parse_kw_col(row['생존후보_키워드'])

    result = []
    
    for kw in confirmed:
        result.extend(apply_rules(kw, is_confirmed=True))
        
    other_pool = list(dict.fromkeys(before + rescued))
    for kw in other_pool:
        if kw in confirmed: continue 
        extracted = apply_rules(kw, is_confirmed=False)
        result.extend(extracted)

    return list(dict.fromkeys(result))  


df_qual['확정키워드_최종'] = df_qual.apply(build_final_keywords, axis=1)

print(f"처리 완료: {len(df_qual)}개 제품")
print(f"확정키워드_최종 평균 키워드 수: {df_qual['확정키워드_최종'].apply(len).mean():.1f}개")

In [ ]:
df_meta = df[['p_name', 'brand', 'p_price', 'p_cap', 'date']].drop_duplicates('p_name')
df_qual = df_qual.merge(df_meta, left_on='제품명', right_on='p_name', how='left').drop(columns='p_name')

print(f"병합 완료: {len(df_qual)}개 제품")
print(f"컬럼: {list(df_qual.columns)}")

In [ ]:
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'eda', 'df_qual_checkpoint.pkl')
df_qual.to_pickle(CHECKPOINT_PATH)
print(f"체크포인트 저장 완료: {CHECKPOINT_PATH}")
print(f"저장 행 수: {len(df_qual)}개 / 컬럼: {list(df_qual.columns)}")

## Step 7. 시각화 및 EDA

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import ast
import pickle
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

BASE_DIR = r'C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework'
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'eda', 'df_qual_checkpoint.pkl')

df_qual = pd.read_pickle(CHECKPOINT_PATH)

print(f"체크포인트 로드 완료: {len(df_qual)}개 제품")
print(f"컬럼: {list(df_qual.columns)}")
df_qual.head()

### Step 6.1. 키워드 빈도 분석 (쓸모없는 키워드 탐색)

In [ ]:
from collections import Counter

all_kws = []
for kw_list in df_qual['확정키워드_최종']:
    if isinstance(kw_list, list):
        all_kws.extend(kw_list)

kw_freq = Counter(all_kws)
df_freq = pd.DataFrame(kw_freq.most_common(), columns=['키워드', '빈도'])
df_freq['등장_제품수'] = df_freq['키워드'].apply(
    lambda kw: df_qual['확정키워드_최종'].apply(
        lambda x: kw in x if isinstance(x, list) else False
    ).sum()
)

print(f"전체 고유 키워드 수: {len(df_freq)}개")
print(f"전체 키워드 등장 횟수: {len(all_kws)}회")
df_freq.head(50)

In [ ]:
FREQ_THRESHOLD = 3

df_low = df_freq[df_freq['빈도'] <= FREQ_THRESHOLD].sort_values('빈도')
print(f"빈도 {FREQ_THRESHOLD} 이하 키워드: {len(df_low)}개")
print(f"전체 고유 키워드 중 {len(df_low)/len(df_freq)*100:.1f}% 차지")
df_low

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top40 = df_freq.head(40)
axes[0].barh(top40['키워드'][::-1], top40['빈도'][::-1], color='steelblue')
axes[0].set_title('상위 40개 키워드 빈도', fontsize=13)
axes[0].set_xlabel('빈도')

axes[1].hist(df_freq['빈도'], bins=50, color='salmon', edgecolor='white')
axes[1].axvline(FREQ_THRESHOLD, color='red', linestyle='--', label=f'임계값 ({FREQ_THRESHOLD})')
axes[1].set_title('키워드 빈도 분포', fontsize=13)
axes[1].set_xlabel('빈도')
axes[1].set_ylabel('키워드 수')
axes[1].legend()

plt.tight_layout()
plt.show()